<a href="https://colab.research.google.com/github/Heashalla/FYP/blob/main/02_Data_Merging_and_Cleaning_ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NHANES Cardiometabolic-Risk Project

## Notebook 02: Data Merging and Cleaning

This notebook selects the required variables from the Excel datasets, merges participant records using SEQN, combines the three survey cycles, and prepares a cleaned dataset for feature and outcome creation.

In [1]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
from pathlib import Path

project_folder = Path("/content/drive/MyDrive/NHANES_Project")
excel_root = project_folder / "raw_excel"

excel_files = sorted(excel_root.rglob("*.xlsx"))

print("Excel folder:", excel_root)
print("Folder exists:", excel_root.exists())
print("Excel files found:", len(excel_files))

for cycle in ["2013_2014", "2015_2016", "2017_2018"]:
    cycle_folder = excel_root / cycle
    cycle_files = sorted(cycle_folder.glob("*.xlsx"))

    print(f"\n{cycle}: {len(cycle_files)} files")

    for file in cycle_files:
        print(" -", file.name)

if len(excel_files) == 30:
    print("\nStatus: All 30 Excel datasets are available.")
else:
    print(f"\nStatus: Expected 30 files but found {len(excel_files)}.")

Excel folder: /content/drive/MyDrive/NHANES_Project/raw_excel
Folder exists: True
Excel files found: 30

2013_2014: 10 files
 - BMX_H.xlsx
 - BPQ_H.xlsx
 - BPX_H.xlsx
 - DEMO_H.xlsx
 - DIQ_H.xlsx
 - DR1TOT_H.xlsx
 - GLU_H.xlsx
 - HDL_H.xlsx
 - PAQ_H.xlsx
 - TRIGLY_H.xlsx

2015_2016: 10 files
 - BMX_I.xlsx
 - BPQ_I.xlsx
 - BPX_I.xlsx
 - DEMO_I.xlsx
 - DIQ_I.xlsx
 - DR1TOT_I.xlsx
 - GLU_I.xlsx
 - HDL_I.xlsx
 - PAQ_I.xlsx
 - TRIGLY_I.xlsx

2017_2018: 10 files
 - BMX_J.xlsx
 - BPQ_J.xlsx
 - BPX_J.xlsx
 - DEMO_J.xlsx
 - DIQ_J.xlsx
 - DR1TOT_J.xlsx
 - GLU_J.xlsx
 - HDL_J.xlsx
 - PAQ_J.xlsx
 - TRIGLY_J.xlsx

Status: All 30 Excel datasets are available.


## 2.1 Check Required Variables for 2013–2014

This section confirms that the variables required for participant identification, predictors, clinical outcome construction and survey-weighted analysis are available in the 2013–2014 Excel datasets.

In [3]:
import pandas as pd
from IPython.display import display

cycle_folder = excel_root / "2013_2014"

required_variables = {
    "DEMO_H": [
        "SEQN",
        "RIAGENDR",
        "RIDAGEYR",
        "SDMVPSU",
        "SDMVSTRA"
    ],
    "BMX_H": [
        "SEQN",
        "BMXWT",
        "BMXHT",
        "BMXWAIST",
        "BMXBMI"
    ],
    "BPX_H": [
        "SEQN",
        "BPXSY1", "BPXDI1",
        "BPXSY2", "BPXDI2",
        "BPXSY3", "BPXDI3",
        "BPXSY4", "BPXDI4"
    ],
    "BPQ_H": [
        "SEQN",
        "BPQ020",
        "BPQ040A"
    ],
    "GLU_H": [
        "SEQN",
        "LBXGLU",
        "WTSAF2YR"
    ],
    "DIQ_H": [
        "SEQN",
        "DIQ050",
        "DIQ070"
    ],
    "TRIGLY_H": [
        "SEQN",
        "LBXTR",
        "WTSAF2YR"
    ],
    "HDL_H": [
        "SEQN",
        "LBDHDD"
    ]
}

availability_records = []
dataset_records = []

for dataset_name, variables in required_variables.items():

    file_path = cycle_folder / f"{dataset_name}.xlsx"

    # Read only a few rows because this stage checks structure
    sample_data = pd.read_excel(
        file_path,
        sheet_name="Raw_Data",
        nrows=5
    )

    available_columns = set(sample_data.columns)

    dataset_records.append({
        "Dataset": dataset_name,
        "File_Exists": file_path.exists(),
        "Total_Columns": len(sample_data.columns),
        "Required_Variables": len(variables),
        "Required_Available": sum(
            variable in available_columns
            for variable in variables
        )
    })

    for variable in variables:
        availability_records.append({
            "Dataset": dataset_name,
            "Variable": variable,
            "Available": variable in available_columns
        })

dataset_summary = pd.DataFrame(dataset_records)
variable_availability = pd.DataFrame(availability_records)

print("2013–2014 DATASET SUMMARY")
print("-" * 70)
display(dataset_summary)

print("\nREQUIRED VARIABLE AVAILABILITY")
print("-" * 70)
display(variable_availability)

missing_variables = variable_availability[
    variable_availability["Available"] == False
]

print("\nVALIDATION")
print("-" * 70)

if missing_variables.empty:
    print("Status: All required 2013–2014 variables are available.")
else:
    print("Status: Some expected variables are unavailable.")
    display(missing_variables)

2013–2014 DATASET SUMMARY
----------------------------------------------------------------------


,Dataset,File_Exists,Total_Columns,Required_Variables,Required_Available
0,DEMO_H,True,47,5,5
1,BMX_H,True,26,5,5
2,BPX_H,True,23,9,9
3,BPQ_H,True,14,3,3
4,GLU_H,True,6,3,3
5,DIQ_H,True,54,3,3
6,TRIGLY_H,True,6,3,3
7,HDL_H,True,3,2,2



REQUIRED VARIABLE AVAILABILITY
----------------------------------------------------------------------


,Dataset,Variable,Available
0,DEMO_H,SEQN,True
1,DEMO_H,RIAGENDR,True
2,DEMO_H,RIDAGEYR,True
3,DEMO_H,SDMVPSU,True
4,DEMO_H,SDMVSTRA,True
5,BMX_H,SEQN,True
6,BMX_H,BMXWT,True
7,BMX_H,BMXHT,True
8,BMX_H,BMXWAIST,True
9,BMX_H,BMXBMI,True



VALIDATION
----------------------------------------------------------------------
Status: All required 2013–2014 variables are available.


## 2.2 Load Selected Variables for 2013–2014

This section loads only the variables required for the main analysis. It checks participant counts, unique participant IDs, duplicates and missing values before merging.

In [4]:
cycle_2013_data = {}
loading_summary = []

for dataset_name, variables in required_variables.items():

    file_path = cycle_folder / f"{dataset_name}.xlsx"

    dataframe = pd.read_excel(
        file_path,
        sheet_name="Raw_Data",
        usecols=variables
    )

    # Store participant ID as a whole-number identifier
    dataframe["SEQN"] = pd.to_numeric(
        dataframe["SEQN"],
        errors="coerce"
    ).astype("Int64")

    cycle_2013_data[dataset_name] = dataframe

    loading_summary.append({
        "Dataset": dataset_name,
        "Rows": len(dataframe),
        "Columns_Selected": len(dataframe.columns),
        "Unique_SEQN": dataframe["SEQN"].nunique(dropna=True),
        "Duplicate_SEQN": int(dataframe["SEQN"].duplicated().sum()),
        "Missing_SEQN": int(dataframe["SEQN"].isna().sum()),
        "Total_Missing_Cells": int(dataframe.isna().sum().sum())
    })

loading_summary_2013 = pd.DataFrame(loading_summary)

print("2013–2014 SELECTED-DATA SUMMARY")
print("-" * 70)
display(loading_summary_2013)

total_duplicates = loading_summary_2013["Duplicate_SEQN"].sum()
total_missing_ids = loading_summary_2013["Missing_SEQN"].sum()

print("\nVALIDATION")
print("-" * 70)

if total_duplicates == 0 and total_missing_ids == 0:
    print("Status: All datasets have valid and unique participant IDs.")
    print("The datasets are ready for participant-level merging.")
else:
    print("Status: Participant-ID problems were detected.")
    print("Total duplicate IDs:", total_duplicates)
    print("Total missing IDs:", total_missing_ids)

2013–2014 SELECTED-DATA SUMMARY
----------------------------------------------------------------------


,Dataset,Rows,Columns_Selected,Unique_SEQN,Duplicate_SEQN,Missing_SEQN,Total_Missing_Cells
0,DEMO_H,10175,5,10175,0,0,0
1,BMX_H,9813,5,9813,0,0,2746
2,BPX_H,9813,9,9813,0,0,33496
3,BPQ_H,6464,3,6464,0,0,4290
4,GLU_H,3329,3,3329,0,0,157
5,DIQ_H,9770,3,9770,0,0,8572
6,TRIGLY_H,3329,3,3329,0,0,183
7,HDL_H,8291,2,8291,0,0,667



VALIDATION
----------------------------------------------------------------------
Status: All datasets have valid and unique participant IDs.
The datasets are ready for participant-level merging.


In [5]:
for dataset_name, dataframe in cycle_2013_data.items():

    print(f"\n{dataset_name}")
    print("-" * 70)

    display(dataframe.head(3))


DEMO_H
----------------------------------------------------------------------


,SEQN,RIAGENDR,RIDAGEYR,SDMVPSU,SDMVSTRA
0,73557,1,69.0,1,112
1,73558,1,54.0,1,108
2,73559,1,72.0,1,109



BMX_H
----------------------------------------------------------------------


,SEQN,BMXWT,BMXHT,BMXBMI,BMXWAIST
0,73557,78.3,171.3,26.7,100.0
1,73558,89.5,176.8,28.6,107.6
2,73559,88.9,175.3,28.9,109.2



BPX_H
----------------------------------------------------------------------


...

,SEQN,DIQ050,DIQ070
0,73557,1.0,1.0
1,73558,1.0,2.0
2,73559,1.0,2.0



TRIGLY_H
----------------------------------------------------------------------


,SEQN,WTSAF2YR,LBXTR
0,73559,142196.890197,51.0
1,73561,142266.006548,75.0
2,73564,134054.109760,64.0



HDL_H
----------------------------------------------------------------------


,SEQN,LBDHDD
0,73557,65.0
1,73558,50.0
2,73559,60.0


In [6]:
import numpy as np

weight_comparison = (
    cycle_2013_data["GLU_H"][["SEQN", "WTSAF2YR"]]
    .merge(
        cycle_2013_data["TRIGLY_H"][["SEQN", "WTSAF2YR"]],
        on="SEQN",
        how="inner",
        suffixes=("_GLU", "_TRIGLY"),
        validate="one_to_one"
    )
)

valid_weight_pairs = weight_comparison.dropna(
    subset=["WTSAF2YR_GLU", "WTSAF2YR_TRIGLY"]
)

weight_matches = np.isclose(
    valid_weight_pairs["WTSAF2YR_GLU"],
    valid_weight_pairs["WTSAF2YR_TRIGLY"],
    rtol=1e-10,
    atol=1e-10
)

print("FASTING-WEIGHT VALIDATION")
print("-" * 70)
print("Participants appearing in both files:", len(weight_comparison))
print("Participants with two available weights:", len(valid_weight_pairs))
print("Matching weight values:", int(weight_matches.sum()))
print("Different weight values:", int((~weight_matches).sum()))

if weight_matches.all():
    print("\nStatus: The fasting weights agree.")
    print("WTSAF2YR will be retained from GLU_H only.")
else:
    print("\nStatus: Some fasting weights differ and require investigation.")

FASTING-WEIGHT VALIDATION
----------------------------------------------------------------------
Participants appearing in both files: 3329
Participants with two available weights: 3329
Matching weight values: 3329
Different weight values: 0

Status: The fasting weights agree.
WTSAF2YR will be retained from GLU_H only.


## 2.5 Merge the 2013–2014 Primary Datasets

The demographic dataset is used as the base. The remaining datasets are joined using SEQN. Left joins preserve all demographic participants so that exclusions and missing-data losses can be documented later.

In [7]:
# Keep the fasting weight from GLU_H only
trigly_for_merge = cycle_2013_data["TRIGLY_H"].drop(
    columns=["WTSAF2YR"]
).copy()

datasets_to_merge = [
    ("BMX_H", cycle_2013_data["BMX_H"]),
    ("BPX_H", cycle_2013_data["BPX_H"]),
    ("BPQ_H", cycle_2013_data["BPQ_H"]),
    ("GLU_H", cycle_2013_data["GLU_H"]),
    ("DIQ_H", cycle_2013_data["DIQ_H"]),
    ("TRIGLY_H", trigly_for_merge),
    ("HDL_H", cycle_2013_data["HDL_H"])
]

merged_2013 = cycle_2013_data["DEMO_H"].copy()
merged_2013["Survey_Cycle"] = "2013-2014"

merge_audit_records = []

for dataset_name, dataset in datasets_to_merge:

    rows_before = len(merged_2013)
    indicator_name = f"_merge_{dataset_name}"

    merged_2013 = merged_2013.merge(
        dataset,
        on="SEQN",
        how="left",
        validate="one_to_one",
        indicator=indicator_name
    )

    matched = int(
        (merged_2013[indicator_name] == "both").sum()
    )

    unmatched = int(
        (merged_2013[indicator_name] == "left_only").sum()
    )

    merged_2013.drop(
        columns=[indicator_name],
        inplace=True
    )

    merge_audit_records.append({
        "Dataset_Added": dataset_name,
        "Rows_Before": rows_before,
        "Matched_Participants": matched,
        "Unmatched_Base_Participants": unmatched,
        "Rows_After": len(merged_2013),
        "Row_Count_Preserved": rows_before == len(merged_2013)
    })

merge_audit_2013 = pd.DataFrame(merge_audit_records)

print("2013–2014 MERGE AUDIT")
print("-" * 70)
display(merge_audit_2013)

print("\nMERGED DATASET VALIDATION")
print("-" * 70)
print("Rows:", merged_2013.shape[0])
print("Columns:", merged_2013.shape[1])
print("Unique SEQN:", merged_2013["SEQN"].nunique())
print("Duplicate SEQN:", merged_2013["SEQN"].duplicated().sum())

if (
    len(merged_2013) == len(cycle_2013_data["DEMO_H"])
    and merged_2013["SEQN"].duplicated().sum() == 0
):
    print("\nStatus: 2013–2014 datasets merged successfully.")
else:
    print("\nStatus: The merged dataset requires investigation.")

print("\nFIRST FIVE MERGED PARTICIPANTS")
display(merged_2013.head())

2013–2014 MERGE AUDIT
----------------------------------------------------------------------


,Dataset_Added,Rows_Before,Matched_Participants,Unmatched_Base_Participants,Rows_After,Row_Count_Preserved
0,BMX_H,10175,9813,362,10175,True
1,BPX_H,10175,9813,362,10175,True
2,BPQ_H,10175,6464,3711,10175,True
3,GLU_H,10175,3329,6846,10175,True
4,DIQ_H,10175,9770,405,10175,True
5,TRIGLY_H,10175,3329,6846,10175,True
6,HDL_H,10175,8291,1884,10175,True



MERGED DATASET VALIDATION
----------------------------------------------------------------------
Rows: 10175
Columns: 26
Unique SEQN: 10175
Duplicate SEQN: 0

Status: 2013–2014 datasets merged successfully.

FIRST FIVE MERGED PARTICIPANTS


,SEQN,RIAGENDR,RIDAGEYR,SDMVPSU,SDMVSTRA,Survey_Cycle,BMXWT,BMXHT,BMXBMI,BMXWAIST,...,BPXSY4,BPXDI4,BPQ020,BPQ040A,WTSAF2YR,LBXGLU,DIQ050,DIQ070,LBXTR,LBDHDD
0,73557,1,69.0,1,112,2013-2014,78.3,171.3,26.7,100.0,...,NaN,NaN,1.0,1.0,NaN,NaN,1.0,1.0,NaN,65.0
1,73558,1,54.0,1,108,2013-2014,89.5,176.8,28.6,107.6,...,NaN,NaN,1.0,2.0,NaN,NaN,1.0,2.0,NaN,50.0
2,73559,1,72.0,1,109,2013-2014,88.9,175.3,28.9,109.2,...,NaN,NaN,1.0,1.0,142196.890197,193.0,1.0,2.0,51.0,60.0
3,73560,1,9.0,2,109,2013-2014,32.2,137.3,17.1,61.0,...,NaN,NaN,NaN,NaN,NaN,NaN,2.0,NaN,NaN,61.0
4,73561,2,73.0,2,116,2013-2014,52.0,162.4,19.7,NaN,...,NaN,NaN,1.0,1.0,142266.006548,107.0,2.0,NaN,75.0,85.0


## 2.6 Load and Merge the 2015–2016 and 2017–2018 Cycles

This section applies the same variable-selection, participant-ID validation, fasting-weight validation and left-join procedure to the remaining two survey cycles.

In [8]:
component_columns = {
    "DEMO": [
        "SEQN", "RIAGENDR", "RIDAGEYR",
        "SDMVPSU", "SDMVSTRA"
    ],
    "BMX": [
        "SEQN", "BMXWT", "BMXHT",
        "BMXWAIST", "BMXBMI"
    ],
    "BPX": [
        "SEQN",
        "BPXSY1", "BPXDI1",
        "BPXSY2", "BPXDI2",
        "BPXSY3", "BPXDI3",
        "BPXSY4", "BPXDI4"
    ],
    "BPQ": [
        "SEQN", "BPQ020", "BPQ040A"
    ],
    "GLU": [
        "SEQN", "LBXGLU", "WTSAF2YR"
    ],
    "DIQ": [
        "SEQN", "DIQ050", "DIQ070"
    ],
    "TRIGLY": [
        "SEQN", "LBXTR", "WTSAF2YR"
    ],
    "HDL": [
        "SEQN", "LBDHDD"
    ]
}

In [9]:
def load_and_merge_cycle(cycle_folder_name, suffix, cycle_label):

    cycle_path = excel_root / cycle_folder_name
    cycle_data = {}
    loading_records = []

    print(f"\nPROCESSING {cycle_label}")
    print("=" * 75)

    # Load and validate the selected variables
    for component, columns in component_columns.items():

        dataset_name = f"{component}_{suffix}"
        file_path = cycle_path / f"{dataset_name}.xlsx"

        header = pd.read_excel(
            file_path,
            sheet_name="Raw_Data",
            nrows=0
        )

        missing_columns = [
            column for column in columns
            if column not in header.columns
        ]

        if missing_columns:
            raise ValueError(
                f"{dataset_name} is missing: {missing_columns}"
            )

        dataframe = pd.read_excel(
            file_path,
            sheet_name="Raw_Data",
            usecols=columns
        )

        dataframe["SEQN"] = pd.to_numeric(
            dataframe["SEQN"],
            errors="coerce"
        ).astype("Int64")

        duplicate_count = int(
            dataframe["SEQN"].duplicated().sum()
        )

        missing_id_count = int(
            dataframe["SEQN"].isna().sum()
        )

        if duplicate_count > 0 or missing_id_count > 0:
            raise ValueError(
                f"{dataset_name} contains invalid participant IDs."
            )

        cycle_data[component] = dataframe

        loading_records.append({
            "Dataset": dataset_name,
            "Rows": len(dataframe),
            "Columns_Selected": len(dataframe.columns),
            "Unique_SEQN": dataframe["SEQN"].nunique(),
            "Duplicate_SEQN": duplicate_count,
            "Missing_SEQN": missing_id_count
        })

        print(
            f"Loaded {dataset_name}: "
            f"{len(dataframe):,} rows"
        )

    loading_audit = pd.DataFrame(loading_records)

    # Compare fasting weights from GLU and TRIGLY
    weight_check = (
        cycle_data["GLU"][["SEQN", "WTSAF2YR"]]
        .merge(
            cycle_data["TRIGLY"][["SEQN", "WTSAF2YR"]],
            on="SEQN",
            how="inner",
            suffixes=("_GLU", "_TRIGLY"),
            validate="one_to_one"
        )
        .dropna(
            subset=["WTSAF2YR_GLU", "WTSAF2YR_TRIGLY"]
        )
    )

    weights_match = np.isclose(
        weight_check["WTSAF2YR_GLU"],
        weight_check["WTSAF2YR_TRIGLY"],
        rtol=1e-10,
        atol=1e-10
    )

    different_weights = int((~weights_match).sum())

    if different_weights > 0:
        raise ValueError(
            f"{cycle_label}: fasting-weight values do not agree."
        )

    print(
        f"Fasting-weight validation: "
        f"{len(weight_check):,} pairs checked; "
        f"{different_weights} differences"
    )

    # Keep WTSAF2YR from GLU only
    trigly_for_merge = cycle_data["TRIGLY"].drop(
        columns=["WTSAF2YR"]
    ).copy()

    merge_items = [
        ("BMX", cycle_data["BMX"]),
        ("BPX", cycle_data["BPX"]),
        ("BPQ", cycle_data["BPQ"]),
        ("GLU", cycle_data["GLU"]),
        ("DIQ", cycle_data["DIQ"]),
        ("TRIGLY", trigly_for_merge),
        ("HDL", cycle_data["HDL"])
    ]

    merged_data = cycle_data["DEMO"].copy()
    merged_data["Survey_Cycle"] = cycle_label

    merge_records = []

    for component, right_data in merge_items:

        rows_before = len(merged_data)
        indicator = f"_merge_{component}"

        merged_data = merged_data.merge(
            right_data,
            on="SEQN",
            how="left",
            validate="one_to_one",
            indicator=indicator
        )

        matched = int(
            (merged_data[indicator] == "both").sum()
        )

        unmatched = int(
            (merged_data[indicator] == "left_only").sum()
        )

        merged_data.drop(
            columns=[indicator],
            inplace=True
        )

        merge_records.append({
            "Dataset_Added": f"{component}_{suffix}",
            "Rows_Before": rows_before,
            "Matched_Participants": matched,
            "Unmatched_Base_Participants": unmatched,
            "Rows_After": len(merged_data),
            "Row_Count_Preserved": (
                rows_before == len(merged_data)
            )
        })

    merge_audit = pd.DataFrame(merge_records)

    print(f"\n{cycle_label} LOADING AUDIT")
    display(loading_audit)

    print(f"\n{cycle_label} MERGE AUDIT")
    display(merge_audit)

    print(f"\n{cycle_label} FINAL VALIDATION")
    print("-" * 75)
    print("Rows:", len(merged_data))
    print("Columns:", len(merged_data.columns))
    print("Duplicate SEQN:", merged_data["SEQN"].duplicated().sum())

    if (
        len(merged_data) == len(cycle_data["DEMO"])
        and merged_data["SEQN"].duplicated().sum() == 0
        and merge_audit["Row_Count_Preserved"].all()
    ):
        print("Status: Cycle merged successfully.")
    else:
        print("Status: Cycle requires investigation.")

    return cycle_data, merged_data, loading_audit, merge_audit

In [10]:
(
    cycle_2015_data,
    merged_2015,
    loading_audit_2015,
    merge_audit_2015
) = load_and_merge_cycle(
    cycle_folder_name="2015_2016",
    suffix="I",
    cycle_label="2015-2016"
)

(
    cycle_2017_data,
    merged_2017,
    loading_audit_2017,
    merge_audit_2017
) = load_and_merge_cycle(
    cycle_folder_name="2017_2018",
    suffix="J",
    cycle_label="2017-2018"
)


PROCESSING 2015-2016
Loaded DEMO_I: 9,971 rows
Loaded BMX_I: 9,544 rows
Loaded BPX_I: 9,544 rows
Loaded BPQ_I: 6,327 rows
Loaded GLU_I: 3,191 rows
Loaded DIQ_I: 9,575 rows
Loaded TRIGLY_I: 3,191 rows
Loaded HDL_I: 8,021 rows
Fasting-weight validation: 3,191 pairs checked; 0 differences

2015-2016 LOADING AUDIT


,Dataset,Rows,Columns_Selected,Unique_SEQN,Duplicate_SEQN,Missing_SEQN
0,DEMO_I,9971,5,9971,0,0
1,BMX_I,9544,5,9544,0,0
2,BPX_I,9544,9,9544,0,0
3,BPQ_I,6327,3,6327,0,0
4,GLU_I,3191,3,3191,0,0
5,DIQ_I,9575,3,9575,0,0
6,TRIGLY_I,3191,3,3191,0,0
7,HDL_I,8021,2,8021,0,0



2015-2016 MERGE AUDIT


,Dataset_Added,Rows_Before,Matched_Participants,Unmatched_Base_Participants,Rows_After,Row_Count_Preserved
0,BMX_I,9971,9544,427,9971,True
1,BPX_I,9971,9544,427,9971,True
2,BPQ_I,9971,6327,3644,9971,True
3,GLU_I,9971,3191,6780,9971,True
4,DIQ_I,9971,9575,396,9971,True
5,TRIGLY_I,9971,3191,6780,9971,True
6,HDL_I,9971,8021,1950,9971,True



2015-2016 FINAL VALIDATION
---------------------------------------------------------------------------
Rows: 9971
Columns: 26
Duplicate SEQN: 0
Status: Cycle merged successfully.

PROCESSING 2017-2018
Loaded DEMO_J: 9,254 rows
Loaded BMX_J: 8,704 rows
Loaded BPX_J: 8,704 rows
Loaded BPQ_J: 6,161 rows
Loaded GLU_J: 3,036 rows
Loaded DIQ_J: 8,897 rows
Loaded TRIGLY_J: 3,036 rows
Loaded HDL_J: 7,435 rows
Fasting-weight validation: 3,036 pairs checked; 0 differences

2017-2018 LOADING AUDIT


,Dataset,Rows,Columns_Selected,Unique_SEQN,Duplicate_SEQN,Missing_SEQN
0,DEMO_J,9254,5,9254,0,0
1,BMX_J,8704,5,8704,0,0
2,BPX_J,8704,9,8704,0,0
3,BPQ_J,6161,3,6161,0,0
4,GLU_J,3036,3,3036,0,0
5,DIQ_J,8897,3,8897,0,0
6,TRIGLY_J,3036,3,3036,0,0
7,HDL_J,7435,2,7435,0,0



2017-2018 MERGE AUDIT


,Dataset_Added,Rows_Before,Matched_Participants,Unmatched_Base_Participants,Rows_After,Row_Count_Preserved
0,BMX_J,9254,8704,550,9254,True
1,BPX_J,9254,8704,550,9254,True
2,BPQ_J,9254,6161,3093,9254,True
3,GLU_J,9254,3036,6218,9254,True
4,DIQ_J,9254,8897,357,9254,True
5,TRIGLY_J,9254,3036,6218,9254,True
6,HDL_J,9254,7435,1819,9254,True



2017-2018 FINAL VALIDATION
---------------------------------------------------------------------------
Rows: 9254
Columns: 26
Duplicate SEQN: 0
Status: Cycle merged successfully.


## 2.9 Combine the 2013–2018 Survey Cycles

This section vertically combines the three merged cycle datasets into one dataset. The Survey_Cycle column preserves the source period of every participant.

In [11]:
cycle_summary_before_combining = pd.DataFrame([
    {
        "Survey_Cycle": "2013-2014",
        "Rows": len(merged_2013),
        "Columns": len(merged_2013.columns),
        "Duplicate_SEQN": int(
            merged_2013["SEQN"].duplicated().sum()
        )
    },
    {
        "Survey_Cycle": "2015-2016",
        "Rows": len(merged_2015),
        "Columns": len(merged_2015.columns),
        "Duplicate_SEQN": int(
            merged_2015["SEQN"].duplicated().sum()
        )
    },
    {
        "Survey_Cycle": "2017-2018",
        "Rows": len(merged_2017),
        "Columns": len(merged_2017.columns),
        "Duplicate_SEQN": int(
            merged_2017["SEQN"].duplicated().sum()
        )
    }
])

print("CYCLE DATASETS BEFORE COMBINING")
print("-" * 70)
display(cycle_summary_before_combining)

# Confirm that all cycles have the same columns
columns_2013 = set(merged_2013.columns)
columns_2015 = set(merged_2015.columns)
columns_2017 = set(merged_2017.columns)

same_columns = (
    columns_2013 == columns_2015 == columns_2017
)

print("\nCOLUMN CONSISTENCY")
print("-" * 70)
print("All cycles contain the same columns:", same_columns)

if not same_columns:
    print(
        "2015–2016 differences:",
        sorted(columns_2013.symmetric_difference(columns_2015))
    )
    print(
        "2017–2018 differences:",
        sorted(columns_2013.symmetric_difference(columns_2017))
    )

CYCLE DATASETS BEFORE COMBINING
----------------------------------------------------------------------


,Survey_Cycle,Rows,Columns,Duplicate_SEQN
0,2013-2014,10175,26,0
1,2015-2016,9971,26,0
2,2017-2018,9254,26,0



COLUMN CONSISTENCY
----------------------------------------------------------------------
All cycles contain the same columns: True


In [12]:
combined_2013_2018 = pd.concat(
    [
        merged_2013,
        merged_2015,
        merged_2017
    ],
    axis=0,
    ignore_index=True
)

expected_rows = (
    len(merged_2013)
    + len(merged_2015)
    + len(merged_2017)
)

cycle_counts = (
    combined_2013_2018["Survey_Cycle"]
    .value_counts()
    .sort_index()
    .rename_axis("Survey_Cycle")
    .reset_index(name="Participants")
)

print("COMBINED DATASET VALIDATION")
print("-" * 70)
print("Expected rows:", expected_rows)
print("Actual rows:", len(combined_2013_2018))
print("Columns:", len(combined_2013_2018.columns))
print(
    "Unique SEQN:",
    combined_2013_2018["SEQN"].nunique()
)
print(
    "Duplicate SEQN:",
    combined_2013_2018["SEQN"].duplicated().sum()
)

print("\nPARTICIPANTS BY SURVEY CYCLE")
display(cycle_counts)

if (
    len(combined_2013_2018) == expected_rows
    and combined_2013_2018["SEQN"].duplicated().sum() == 0
):
    print(
        "\nStatus: All three survey cycles were "
        "combined successfully."
    )
else:
    print(
        "\nStatus: The combined dataset requires investigation."
    )

print("\nFIRST FIVE COMBINED RECORDS")
display(combined_2013_2018.head())

COMBINED DATASET VALIDATION
----------------------------------------------------------------------
Expected rows: 29400
Actual rows: 29400
Columns: 26
Unique SEQN: 29400
Duplicate SEQN: 0

PARTICIPANTS BY SURVEY CYCLE


,Survey_Cycle,Participants
0,2013-2014,10175
1,2015-2016,9971
2,2017-2018,9254



Status: All three survey cycles were combined successfully.

FIRST FIVE COMBINED RECORDS


,SEQN,RIAGENDR,RIDAGEYR,SDMVPSU,SDMVSTRA,Survey_Cycle,BMXWT,BMXHT,BMXBMI,BMXWAIST,...,BPXSY4,BPXDI4,BPQ020,BPQ040A,WTSAF2YR,LBXGLU,DIQ050,DIQ070,LBXTR,LBDHDD
0,73557,1,69.0,1,112,2013-2014,78.3,171.3,26.7,100.0,...,NaN,NaN,1.0,1.0,NaN,NaN,1.0,1.0,NaN,65.0
1,73558,1,54.0,1,108,2013-2014,89.5,176.8,28.6,107.6,...,NaN,NaN,1.0,2.0,NaN,NaN,1.0,2.0,NaN,50.0
2,73559,1,72.0,1,109,2013-2014,88.9,175.3,28.9,109.2,...,NaN,NaN,1.0,1.0,142196.890197,193.0,1.0,2.0,51.0,60.0
3,73560,1,9.0,2,109,2013-2014,32.2,137.3,17.1,61.0,...,NaN,NaN,NaN,NaN,NaN,NaN,2.0,NaN,NaN,61.0
4,73561,2,73.0,2,116,2013-2014,52.0,162.4,19.7,NaN,...,NaN,NaN,1.0,1.0,142266.006548,107.0,2.0,NaN,75.0,85.0


## 2.11 Save the Merged Datasets

This section saves each cycle-level merged dataset and the combined 2013–2018 dataset as Excel files. These files represent the complete merged data before cleaning or participant exclusions.

In [14]:
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter

processed_folder = project_folder / "processed_excel"
merged_cycle_folder = processed_folder / "merged_by_cycle"
reports_folder = project_folder / "reports"

processed_folder.mkdir(parents=True, exist_ok=True)
merged_cycle_folder.mkdir(parents=True, exist_ok=True)
reports_folder.mkdir(parents=True, exist_ok=True)


def save_merged_excel(dataframe, output_path, sheet_name):

    with pd.ExcelWriter(
        output_path,
        engine="openpyxl"
    ) as writer:

        dataframe.to_excel(
            writer,
            sheet_name=sheet_name,
            index=False
        )

        worksheet = writer.book[sheet_name]

        worksheet.freeze_panes = "A2"
        worksheet.auto_filter.ref = worksheet.dimensions
        worksheet.row_dimensions[1].height = 30

        header_fill = PatternFill(
            fill_type="solid",
            fgColor="1F4E78"
        )

        for cell in worksheet[1]:
            cell.font = Font(
                bold=True,
                color="FFFFFF"
            )
            cell.fill = header_fill
            cell.alignment = Alignment(
                horizontal="center",
                vertical="center",
                wrap_text=True
            )

        for column_number, column_name in enumerate(
            dataframe.columns,
            start=1
        ):
            width = min(max(len(str(column_name)) + 2, 12), 22)

            worksheet.column_dimensions[
                get_column_letter(column_number)
            ].width = width


merged_output_files = {
    merged_cycle_folder / "NHANES_2013_2014_Merged.xlsx":
        (merged_2013, "Merged_2013_2014"),

    merged_cycle_folder / "NHANES_2015_2016_Merged.xlsx":
        (merged_2015, "Merged_2015_2016"),

    merged_cycle_folder / "NHANES_2017_2018_Merged.xlsx":
        (merged_2017, "Merged_2017_2018"),

    processed_folder / "NHANES_2013_2018_Combined_Merged.xlsx":
        (combined_2013_2018, "Combined_Merged")
}

print("SAVING MERGED DATASETS")
print("-" * 70)

for output_path, dataset_information in merged_output_files.items():

    dataframe, sheet_name = dataset_information

    save_merged_excel(
        dataframe=dataframe,
        output_path=output_path,
        sheet_name=sheet_name
    )

    print(
        f"Saved: {output_path.name} "
        f"({len(dataframe):,} rows × "
        f"{len(dataframe.columns)} columns)"
    )

SAVING MERGED DATASETS
----------------------------------------------------------------------
Saved: NHANES_2013_2014_Merged.xlsx (10,175 rows × 26 columns)
Saved: NHANES_2015_2016_Merged.xlsx (9,971 rows × 26 columns)
Saved: NHANES_2017_2018_Merged.xlsx (9,254 rows × 26 columns)
Saved: NHANES_2013_2018_Combined_Merged.xlsx (29,400 rows × 26 columns)


In [15]:
merge_audit_file = reports_folder / "NHANES_Merge_Audit.xlsx"

with pd.ExcelWriter(
    merge_audit_file,
    engine="openpyxl"
) as writer:

    cycle_summary_before_combining.to_excel(
        writer,
        sheet_name="Cycle Summary",
        index=False
    )

    loading_summary_2013.to_excel(
        writer,
        sheet_name="2013 Loading",
        index=False
    )

    merge_audit_2013.to_excel(
        writer,
        sheet_name="2013 Merge",
        index=False
    )

    loading_audit_2015.to_excel(
        writer,
        sheet_name="2015 Loading",
        index=False
    )

    merge_audit_2015.to_excel(
        writer,
        sheet_name="2015 Merge",
        index=False
    )

    loading_audit_2017.to_excel(
        writer,
        sheet_name="2017 Loading",
        index=False
    )

    merge_audit_2017.to_excel(
        writer,
        sheet_name="2017 Merge",
        index=False
    )

    cycle_counts.to_excel(
        writer,
        sheet_name="Combined Counts",
        index=False
    )

    for worksheet in writer.book.worksheets:

        worksheet.freeze_panes = "A2"
        worksheet.auto_filter.ref = worksheet.dimensions
        worksheet.row_dimensions[1].height = 30

        for cell in worksheet[1]:
            cell.font = Font(
                bold=True,
                color="FFFFFF"
            )
            cell.fill = PatternFill(
                fill_type="solid",
                fgColor="1F4E78"
            )
            cell.alignment = Alignment(
                horizontal="center",
                vertical="center",
                wrap_text=True
            )

        for column_cells in worksheet.columns:
            column_letter = column_cells[0].column_letter

            maximum_length = max(
                len(str(cell.value))
                if cell.value is not None else 0
                for cell in column_cells
            )

            worksheet.column_dimensions[column_letter].width = min(
                maximum_length + 2,
                30
            )

print("\nMerge audit saved:", merge_audit_file)


Merge audit saved: /content/drive/MyDrive/NHANES_Project/reports/NHANES_Merge_Audit.xlsx


In [16]:
print("MERGED OUTPUT VALIDATION")
print("-" * 70)

for output_path in merged_output_files:

    print(
        f"{output_path.name}: "
        f"{'Found' if output_path.exists() else 'Missing'}"
    )

print(
    f"{merge_audit_file.name}: "
    f"{'Found' if merge_audit_file.exists() else 'Missing'}"
)

all_outputs_exist = (
    all(path.exists() for path in merged_output_files)
    and merge_audit_file.exists()
)

if all_outputs_exist:
    print("\nStatus: All merged datasets and audits were saved.")
else:
    print("\nStatus: One or more output files are missing.")

MERGED OUTPUT VALIDATION
----------------------------------------------------------------------
NHANES_2013_2014_Merged.xlsx: Found
NHANES_2015_2016_Merged.xlsx: Found
NHANES_2017_2018_Merged.xlsx: Found
NHANES_2013_2018_Combined_Merged.xlsx: Found
NHANES_Merge_Audit.xlsx: Found

Status: All merged datasets and audits were saved.
